In [ ]:

!pip install -q ydf
import gc,json,time
from pathlib import Path
import numpy as np,pandas as pd,ydf
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
ID="id";TARGET="health_condition";C=np.array(["at-risk","fit","unhealthy"]);SEED=2027;FOLDS=5
base=Path("/kaggle/input/competitions/playground-series-s6e7");tr=pd.read_csv(base/"train.csv");te=pd.read_csv(base/"test.csv");y=tr[TARGET].to_numpy();t0=time.time()
def fe(d):
 z=d.drop(columns=[ID],errors="ignore").copy();s=z.sleep_duration
 z["sleep_lt6"]=np.where(s.isna(),np.nan,(s<6).astype(float));z["sleep_lt7"]=np.where(s.isna(),np.nan,(s<7).astype(float))
 z["key_missing_count"]=z[["sleep_duration","stress_level","physical_activity_level"]].isna().sum(1)
 z["rule_unhealthy"]=((s<6)&z.stress_level.eq("high")).astype(float);z["rule_fit"]=((s>=7)&z.stress_level.eq("low")&z.physical_activity_level.eq("active")).astype(float)
 return z
A=fe(tr);B=fe(te);oof=np.zeros((len(A),3),np.float32);pred=np.zeros((len(B),3));fs=[]
weights={c:len(y)/(3*(y==c).sum()) for c in C};skf=StratifiedKFold(FOLDS,shuffle=True,random_state=SEED)
for f,(a,b) in enumerate(skf.split(A,y),1):
 learner=ydf.GradientBoostedTreesLearner(label=TARGET,task=ydf.Task.CLASSIFICATION,num_trees=1800,max_depth=8,shrinkage=.04,subsample=.8,min_examples=30,l2_regularization=2.0,class_weights=weights,random_seed=SEED+f)
 model=learner.train(A.iloc[a]);oof[b]=model.predict(A.iloc[b]);pred+=model.predict(B)/FOLDS
 s=balanced_accuracy_score(y[b],C[oof[b].argmax(1)]);fs.append(float(s));print(f,s);del model;gc.collect()
score=float(balanced_accuracy_score(y,C[oof.argmax(1)]));od=pd.DataFrame({ID:tr[ID]});td=pd.DataFrame({ID:te[ID]})
for j,c in enumerate(C):od[c]=oof[:,j];td[c]=pred[:,j]
od.to_csv("oof_preds.csv",index=False);td.to_csv("test_preds.csv",index=False);pd.DataFrame({ID:te[ID],TARGET:C[pred.argmax(1)]}).to_csv("submission.csv",index=False)
Path("training_summary.json").write_text(json.dumps({"experiment":"ydf_gbt","fold_scores":fs,"oof_balanced_accuracy":score,"elapsed_minutes":(time.time()-t0)/60},indent=2));print(score)
